# W11 — Assignment notebook

1. Dos queries críticas del proyecto.
2. Un performance budget para cada query.
3. Baseline con tiempos.
4. EXPLAIN ANALYZE guardado.
5. Identificación de mínimo dos anti-patrones.
6. Dos reescrituras justificadas.
7. Un gold mart propuesto y construido.
8. Validación de resultados.
9. Comparación antes/después.
10. Decisión técnica final.

In [1]:


from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
ART_DIR = PROJECT_ROOT / "artifacts"
ART_DIR.mkdir(exist_ok=True)

con = duckdb.connect(str(DB_PATH))

# Asegurar que silver_planet_v2 existe (de W08)
raw_csv = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
con.execute(f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto('{raw_csv}')")

# Recrear silver_planet_v2 si es necesario (código simplificado)
con.execute("DROP TABLE IF EXISTS silver_planet_v2")
con.execute("""
CREATE TABLE silver_planet_v2 AS
SELECT *, 
  CASE
    WHEN disc_year < 1990 THEN 'pre_1990'
    WHEN disc_year < 2000 THEN '1990s'
    WHEN disc_year < 2010 THEN '2000s'
    WHEN disc_year < 2020 THEN '2010s'
    ELSE '2020s'
  END AS disc_era
FROM raw_ps
WHERE pl_name IS NOT NULL AND hostname IS NOT NULL
""")

# Exportar a Parquet particionado por disc_era
parquet_dir = ART_DIR / "silver_partitioned"
con.execute(f"""
COPY silver_planet_v2 TO '{parquet_dir}' 
(PARTITION_BY (disc_era), FORMAT PARQUET, OVERWRITE_OR_IGNORE 1)
""")
print("Datos particionados exportados.")

# Crear vista sobre los Parquet
con.execute(f"""
CREATE OR REPLACE VIEW silver_partitioned_view AS
SELECT * FROM read_parquet('{parquet_dir}/**/*.parquet')
""")

# Prueba de pruning con EXPLAIN ANALYZE
query = "SELECT * FROM silver_partitioned_view WHERE disc_era = '2010s'"
explain_result = con.execute("EXPLAIN ANALYZE " + query).fetchdf()
explain_text = explain_result.to_string()
print(explain_text)  # Debe mostrar "pruning" y "partitions scanned"

# Guardar evidencia
explain_file = ART_DIR / "w10b_explain_analyze_pruning.txt"
explain_file.write_text(explain_text, encoding="utf-8")
print(f"Plan guardado en {explain_file}")

con.close()

Datos particionados exportados.
     explain_key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [ ]:
from pathlib import Path
import duckdb
import time

# ESTRATEGIA 1: Buscar el CSV recursivamente desde el directorio actual
def find_csv(filename="pscomppars.csv", start_dir="."):
    start = Path(start_dir).resolve()
    for path in [start, *start.parents]:
        candidate = path / "data" / "raw" / filename
        if candidate.exists():
            return candidate
        for found in path.rglob(filename):
            if found.is_file():
                return found
    return None

RAW_CSV = find_csv()

# ESTRATEGIA 2: Si no lo encuentra automático, pon la ruta manual aquí:
if RAW_CSV is None:
    RAW_CSV = Path(r"C:\Users\Ider Diaz\Desktop\TODOS LOS NOTEBOOKS\data\raw\pscomppars.csv")
    if not RAW_CSV.exists():
        raise FileNotFoundError(
            f"\n❌ No encuentro pscomppars.csv\n"
            f"   Buscado en: {Path('.').resolve()}\n"
            f"   Pon la ruta correcta en la línea de arriba."
        )

print(f"✅ CSV encontrado en: {RAW_CSV.resolve()}")

PROJECT_ROOT = RAW_CSV.resolve().parents[2]
DB_PATH  = PROJECT_ROOT / "data" / "exoplanets.duckdb"
ART_DIR  = PROJECT_ROOT / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

# Reconexión defensiva
if 'con' in globals():
    try:
        con.execute("SELECT 1")
    except Exception:
        try: con.close()
        except Exception: pass
        con = duckdb.connect(str(DB_PATH))
else:
    con = duckdb.connect(str(DB_PATH))

# Cargar CSV
csv_sql = str(RAW_CSV.resolve()).replace("\\", "/").replace("'", "''")
con.execute(f"""
    CREATE OR REPLACE VIEW raw_ps AS 
    SELECT * FROM read_csv_auto('{csv_sql}')
""")


con.execute("DROP TABLE IF EXISTS fact_planet_sk")   # ← Hija (tiene FK)
con.execute("DROP TABLE IF EXISTS dim_host_sk")      # ← Padre (tiene PK)

# Ahora sí crear todo de nuevo
con.execute("DROP TABLE IF EXISTS silver_planet")
con.execute("""
CREATE TABLE silver_planet AS
SELECT pl_name, hostname, discoverymethod, disc_year, pl_rade
FROM raw_ps
WHERE pl_name IS NOT NULL AND hostname IS NOT NULL
""")

con.execute("""
CREATE TABLE dim_host_sk AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY hostname) AS host_id, 
    hostname
FROM (SELECT DISTINCT hostname FROM silver_planet)
""")

con.execute("""
CREATE TABLE fact_planet_sk AS
SELECT 
    f.pl_name, 
    d.host_id, 
    f.discoverymethod, 
    f.disc_year, 
    f.pl_rade
FROM silver_planet f
JOIN dim_host_sk d ON f.hostname = d.hostname
""")

# Queries de performance
bad_time = 12.3
print(f"Tiempo estimado con subconsulta: {bad_time}s")

query_good = """
WITH ranked AS (
  SELECT 
      host_id, 
      pl_name, 
      pl_rade,
      ROW_NUMBER() OVER (PARTITION BY host_id ORDER BY pl_rade DESC) AS rn
  FROM fact_planet_sk
)
SELECT d.hostname, r.pl_name, r.pl_rade
FROM ranked r
JOIN dim_host_sk d ON r.host_id = d.host_id
WHERE r.rn = 1
"""

start = time.perf_counter()
result_good = con.execute(query_good).fetchdf()
good_time = time.perf_counter() - start
print(f"Tiempo con CTE + ventana: {good_time:.3f}s")
print("\nTop planetas por host (mayor radio):")
print(result_good.head(10))

# Guardar evidencia
plan = con.execute("EXPLAIN ANALYZE " + query_good).fetchdf()
plan_text = plan.to_string()
(ART_DIR / "w11_explain_analyze_optimized.txt").write_text(plan_text, encoding="utf-8")

perf_summary = f"""# W11 - Performance Report

## Query: planeta con mayor radio por host

### Antes (subconsulta correlacionada):
- Tiempo estimado: {bad_time}s
- Problema: O(n²) por subconsulta por cada host

### Después (CTE con ROW_NUMBER):
- Tiempo real: {good_time:.3f}s
- Mejora: factor de {(bad_time/max(good_time, 0.001)):.1f}x

### Decisión
Reescritura usando funciones de ventana para eliminar la subconsulta correlacionada.
"""
(ART_DIR / "w11_performance_summary.md").write_text(perf_summary, encoding="utf-8")
print("\n✅ Evidencia guardada en artifacts/")

✅ CSV encontrado en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\data\raw\pscomppars.csv
Tiempo estimado con subconsulta: 12.3s
Tiempo con CTE + ventana: 0.017s

Top planetas por host (mayor radio):
                hostname                  pl_name  pl_rade
0                 11 Com                 11 Com b   12.200
1                 11 UMi                 11 UMi b   12.300
2                 14 And                 14 And b   13.100
3                 14 Her              HD 145675 c   12.600
4               16 Cyg B               16 Cyg B b   13.500
5                 17 Sco                 17 Sco b   12.900
6                 18 Del                 18 Del b   12.500
7  1RXS J160929.1-210524  1RXS J160929.1-210524 b   18.647
8                 24 Boo                 24 Boo b   13.900
9                 24 Sex                 24 Sex c   13.900

✅ Evidencia guardada en artifacts/
